## Business Understanding

**Identifiers & Metadata**

application_id: A unique identifier for each individual loan application.

customer_id: A unique identifier for each customer. A single customer may have multiple applications.

application_date: The date on which the loan application was submitted.

data_batch_id: An identifier for the data processing batch this record belongs to.

**Loan Characteristics**

loan_amount_requested: The principal amount of the loan requested by the applicant.

loan_amount_usd: The requested loan amount converted to US Dollars for standardization.

loan_tenure_months: The duration of the loan repayment period in months.

interest_rate_offered: The annual interest rate offered for the loan.

purpose_of_loan: The stated reason for seeking the loan.

loan_type_*: A set of binary columns indicating the specific type of loan product.

**Applicant Financial Profile**

employment_status: The applicant's current employment situation.

monthly_income: The applicant's stated gross monthly income.

yearly_income: The applicant's stated gross annual income.

annual_bonus: The applicant's declared annual bonus amount.

cibil_score: A credit score (e.g., from CIBIL) representing the applicant's creditworthiness and history. Higher scores indicate better credit health.

existing_emis_monthly: The total amount of Equated Monthly Installments (EMIs) the applicant is currently paying for other existing loans.

debt_to_income_ratio: This ratio helps assess an applicant's ability to manage monthly payments.

credit_utilization_ratio: The ratio of the applicant's outstanding credit card debt to their total credit card limit.

**Applicant Demographics & Personal Information**

applicant_age: The age of the applicant in years at the time of application.

gender_*: A set of one-hot encoded binary columns representing the applicant's gender.

property_ownership_status: The applicant's housing situation.

residential_address: The applicant's provided residential address (likely anonymized or generalized).

number_of_dependents: The number of people financially dependent on the applicant.

**Target Variable**

fraud_flag: This is the key target variable for prediction. It's a binary indicator where 1 signifies a fraudulent application and 0 signifies a legitimate application.

# Importing Libraries

In [1]:
# basic libraries
import pandas as pd
import numpy as np

# data preparation libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold

# prediction models libraries
from sklearn.metrics import classification_report
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn import linear_model
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier

# Train Dataset

## Data Treatment

In [9]:
df_train = pd.read_csv('train.csv')
df_train = df_train.drop(columns=['Unnamed: 0', 'data_batch_id']) ##removing first column, that looks just an random id
# keep only distinct rows, removing duplicated ones
df_train = df_train.drop_duplicates()

# drop dates
df_train = df_train.drop(columns='application_date')

# removing unwanted object features
categ_cols = ['purpose_of_loan', 'employment_status', 'property_ownership_status']
df_train[categ_cols] = df_train[categ_cols].astype('category')
df_train = df_train.select_dtypes(exclude='object')

# capitalizing category features
for column in categ_cols:
    df_train[column] = df_train[column].str.capitalize()
    df_train[column] = df_train[column].str.strip()

# removing untrustable dummy columns
dummyDrop = [col for col in df_train.columns if 'loan_type' in col]
df_train = df_train.drop(columns=dummyDrop)

## Outliers Removal

In [10]:
# removing outliers by the loan tenure months
q1 = df_train['loan_tenure_months'].quantile(0.25)
q3 = df_train['loan_tenure_months'].quantile(0.75)
iqr = q3-q1
upper_bound = q3 + iqr*1.5

# removing outliers
df_train = df_train[df_train['loan_tenure_months']<=upper_bound]

## Handling Missing Values

### Gender Imputation

In [11]:
## gender imputation through KNN
def gender_code(row):
    if row['gender_Male'] == 1:
        return 1
    elif row['gender_Other'] == 1:
        return 0
    else:
        None #columns as neither male or other will be treated as unkown

df_train['gender_code'] = df_train.apply(gender_code, axis=1)

# defining known and unknown gender
df_known = df_train[df_train['gender_code'].notna()]
df_unknown = df_train[df_train['gender_code'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['gender_code']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_train.loc[df_train['gender_code'].isna(), 'gender_code'] = knn.predict(x_test_scaled)

# recreating the gender label, where "0" is man and "1" is woman
gender_drop = ['gender_Male', 'gender_Other']
df_train = df_train.drop(columns=gender_drop)
df_train = df_train.rename(columns={'gender_code': 'gender_male'})

# 0: woman ; 1: man
df_train['gender_male'].value_counts()

gender_male
0.0    20096
1.0    19541
Name: count, dtype: int64

### Number of Dependents Imputation

In [12]:
## gender imputation through KNN

# defining known and unknown gender
df_known = df_train[df_train['number_of_dependents'].notna()]
df_unknown = df_train[df_train['number_of_dependents'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['number_of_dependents']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_train.loc[df_train['number_of_dependents'].isna(), 'number_of_dependents'] = knn.predict(x_test_scaled)

## Dropping

In [13]:
# monthly income: removed since it has missing values and we already have the yearly income, with same effect
# cibil score: removed since it's perfectly normal, so likely it's not a real feature
# loan amount requested: removed since we have the same feature in dollars, to avoid redundance
df_train = df_train.drop(columns=['monthly_income', 'cibil_score', 'loan_amount_requested'])

## Dummying

In [14]:
# transforming categorical features into dummy ones, removing one of the categories to avoid colinearility
df_train = pd.get_dummies(df_train, columns=categ_cols, drop_first=True, dtype=int)

## Preparing dataset for the models

In [15]:
# setting target variable
y = df_train['fraud_flag']

# setting feature columns
x = df_train.drop(columns='fraud_flag')

# setting dummy columns
dummy_cols = [col for col in x if any(sub in col for sub in categ_cols) or col == 'gender_male']
not_dummy_cols = [col for col in x if col not in dummy_cols] 

# scaling feature columns but dummy ones
scaler = StandardScaler()
x_scaled = pd.DataFrame(
    scaler.fit_transform(x[not_dummy_cols]), #train the model and scales at same time
    columns=not_dummy_cols, 
    index=x.index)

# concatenating the dummy cols and the scaled numeric features
x_final = pd.concat([x_scaled, x[dummy_cols]], axis=1)

# spliting train and test
x_train, x_test, y_train, y_test = train_test_split(x_final, y, test_size=0.2, random_state=42, stratify=y)

# smoted scaled 
smote = SMOTE(sampling_strategy='minority')
x_train_SMOTE, y_smote = smote.fit_resample(x_train, y_train) #smotes train only

# Model Training and Test

## Logistic Regression (Simple)

In [ ]:
# creating model
log_reg_model = linear_model.LogisticRegression(random_state=42, class_weight='balanced')

# parameters Grid
param_grid = [
    {
        'penalty':['l1','l2','elasticnet','none'], 
        'C': np.logspace(-4,4,20), 
        'solver': ['lbfgs','newton-cg','liblinear','sag','saga'], 
        'max_iter': [2000,3000]}]

# executing GridSearch
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_log_reg_model = grid_search.best_estimator_ #already returns the best model trained

# prediction
y_pred = best_log_reg_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred), '\n')
print("Classification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results = dict()

results['Logistic Regression'] = classification_report(y_test, y_pred, output_dict=True)

Confusion Matrix:
col_0          0
fraud_flag      
0           6858
1           1070 

Classification Report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      6858
           1       0.00      0.00      0.00      1070

    accuracy                           0.87      7928
   macro avg       0.43      0.50      0.46      7928
weighted avg       0.75      0.87      0.80      7928



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [ ]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

## Logistic Regression w/ Stepwise

In [128]:
# setting stepwise
sfs_log_reg = SequentialFeatureSelector(
    best_log_reg_model,
    scoring='accuracy',
    cv=None
)

# applying the stepwise model
selected_features = sfs_log_reg.fit(x_train, y_train)

# selecting the stepwised features
x_train_stepwised = x_train[selected_features.get_feature_names_out()]
x_test_stepwised = x_test[selected_features.get_feature_names_out()]

# parameters grid
param_grid = [
    {
        'penalty':['l1','l2','elasticnet','none'], 
        'C': np.logspace(-4,4,20), 
        'solver': ['lbfgs','newton-cg','liblinear','sag','saga'], 
        'max_iter': [2000,3000]}]

# setting GridSearch model
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# applying the GridSearch model
grid_search.fit(x_train_stepwised, y_train)
best_log_reg_model = grid_search.best_estimator_

# prediction
y_pred = best_log_reg_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Logistic Regression w/ Stepwise'] = classification_report(y_test, y_pred, output_dict=True)

# model with best accuracy
x_stepwise_best_model = x_train_stepwised.columns
best_model = best_log_reg_model

Fitting 5 folds for each of 800 candidates, totalling 4000 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The ma

Confusion Matrix:
col_0          0
fraud_flag      
0           6858
1           1070

Classification Report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      6858
           1       0.00      0.00      0.00      1070

    accuracy                           0.87      7928
   macro avg       0.43      0.50      0.46      7928
weighted avg       0.75      0.87      0.80      7928



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
2600 fits failed out of a total of 4000.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
200 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  F

In [129]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.0001 
 solver liblinear 
 max_iter 2000


## Logistic Regression w/ Stepwise and SMOTE resampling

In [130]:
# stepwising after SMOTE
selected_features_SMOTE = sfs_log_reg.fit(x_train_SMOTE, y_smote) #select features after SMOTE
x_stepwised_SMOTE = x_train_SMOTE[selected_features_SMOTE.get_feature_names_out()]

# selecting the stepwise columns for the test sample
x_test_stepwised = x_test[selected_features_SMOTE.get_feature_names_out()]

# parameters Grid
param_grid = [
    {
        'penalty':['l1','l2','elasticnet','none'], 
        'C': np.logspace(-4,4,20), 
        'solver': ['lbfgs','newton-cg','liblinear','sag','saga'], 
        'max_iter': [2000,3000]}]

# executing GridSearch
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# applying the grid search
grid_search.fit(x_stepwised_SMOTE, y_smote)
best_log_reg_model = grid_search.best_estimator_

# the smoted data should be used only for training, not for tests
y_pred = best_log_reg_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Logistic Regression w/ Stepwise and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 800 candidates, totalling 4000 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           2830  4028
1            465   605

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.41      0.56      6858
           1       0.13      0.57      0.21      1070

    accuracy                           0.43      7928
   macro avg       0.49      0.49      0.38      7928
weighted avg       0.76      0.43      0.51      7928



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
2600 fits failed out of a total of 4000.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
200 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  F

In [131]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.0018329807108324356 
 solver saga 
 max_iter 2000


## KNN (Simple)

In [16]:
# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
knn_model = KNeighborsClassifier(weights='distance')
grid_search=GridSearchCV(knn_model, scoring='accuracy', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_train, y_train)
knn_best_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = knn_best_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors})'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0
fraud_flag      
0           6858
1           1070

Classification Report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      6858
           1       0.00      0.00      0.00      1070

    accuracy                           0.87      7928
   macro avg       0.43      0.50      0.46      7928
weighted avg       0.75      0.87      0.80      7928



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

NameError: name 'results' is not defined

## KNN w/ Stepwise

In [17]:
# defining the stepwise model
sfs_knn = SequentialFeatureSelector(
    knn_model,
    scoring='accuracy',
    cv=None
)

# applying the stepwise model
selected_features = sfs_knn.fit(x_train, y_train)

# selecting the stepwised features
x_train_stepwised = x_train[selected_features.get_feature_names_out()]
x_test_stepwised = x_test[selected_features.get_feature_names_out()]

# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
grid_search=GridSearchCV(knn_model, scoring='accuracy', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_train_stepwised, y_train)
knn_best_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = knn_best_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors}) w/ Stepwise'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0
fraud_flag      
0           6858
1           1070

Classification Report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      6858
           1       0.00      0.00      0.00      1070

    accuracy                           0.87      7928
   macro avg       0.43      0.50      0.46      7928
weighted avg       0.75      0.87      0.80      7928



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

NameError: name 'results' is not defined

## KNN w/ Stepwise and SMOTE

In [18]:
# stepwising after SMOTE
selected_features_SMOTE = sfs_knn.fit(x_train_SMOTE, y_smote) #select features after SMOTE
x_stepwised_SMOTE = x_train_SMOTE[selected_features_SMOTE.get_feature_names_out()]

# selecting the stepwise columns for the test sample
x_test_stepwised = x_test[x_stepwised_SMOTE.columns]

# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
grid_search=GridSearchCV(knn_model, scoring='accuracy', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_stepwised_SMOTE, y_smote)
knn_best_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = knn_best_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors}) w/ Stepwise and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6663  195
1           1036   34

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      6858
           1       0.15      0.03      0.05      1070

    accuracy                           0.84      7928
   macro avg       0.51      0.50      0.48      7928
weighted avg       0.77      0.84      0.80      7928



NameError: name 'results' is not defined

## Decision Tree

In [135]:
# loading the decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# training the decision tree model
dtree_model.fit(x_train, y_train)

# predicting
y_pred = dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree'] = classification_report(y_test, y_pred, output_dict=True)

Confusion Matrix:
col_0          0     1
fraud_flag            
0           5723  1135
1            874   196

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.83      0.85      6858
           1       0.15      0.18      0.16      1070

    accuracy                           0.75      7928
   macro avg       0.51      0.51      0.51      7928
weighted avg       0.77      0.75      0.76      7928



## Decision Tree w/ Grid Search

In [136]:
# setting the initial hyperparameters
param_grid = {
    'max_depth': [30, 35, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# loading the grid search decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=dtree_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    error_score='raise'
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_dtree_model = grid_search.best_estimator_

# prediction
y_pred = best_dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree w/ GridSearch'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6200  658
1            972   98

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      6858
           1       0.13      0.09      0.11      1070

    accuracy                           0.79      7928
   macro avg       0.50      0.50      0.50      7928
weighted avg       0.77      0.79      0.78      7928



In [137]:
print(
    'max_depth', best_dtree_model.max_depth, '\n',
    'min_samples_splt', best_dtree_model.min_samples_split, '\n',
    'min_samples_leaf', best_dtree_model.min_samples_leaf)

max_depth 30 
 min_samples_splt 10 
 min_samples_leaf 4


## Decision Tree w/ Grid Search and SMOTE

In [138]:
# setting the initial hyperparameters
param_grid = {
    'max_depth': [30, 35, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# loading the grid search decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=dtree_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train_SMOTE, y_smote)
best_dtree_model = grid_search.best_estimator_

# prediction
y_pred = best_dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree w/ GridSearch and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5853  1005
1            889   181

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      6858
           1       0.15      0.17      0.16      1070

    accuracy                           0.76      7928
   macro avg       0.51      0.51      0.51      7928
weighted avg       0.77      0.76      0.77      7928



In [139]:
print(
    'max_depth', best_dtree_model.max_depth, '\n',
    'min_samples_splt', best_dtree_model.min_samples_split, '\n',
    'min_samples_leaf', best_dtree_model.min_samples_leaf)

max_depth 30 
 min_samples_splt 5 
 min_samples_leaf 2


## Random Forest w/ GridSearch

In [140]:
# setting the initial hyperparameters
param_grid = {
    'n_estimators': [50, 70],
    'max_depth': [15, 20],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# setting the random forest model
rand_for = RandomForestClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=rand_for,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_rand_for_model = grid_search.best_estimator_

# prediction
y_pred = best_rand_for_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Random Forest w/ GridSearch'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Confusion Matrix:
col_0          0
fraud_flag      
0           6858
1           1070

Classification Report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      6858
           1       0.00      0.00      0.00      1070

    accuracy                           0.87      7928
   macro avg       0.43      0.50      0.46      7928
weighted avg       0.75      0.87      0.80      7928



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [141]:
print(
    'n_estimators', best_rand_for_model.n_estimators, '\n'
    'max_depth', best_rand_for_model.max_depth, '\n',
    'min_samples_splt', best_rand_for_model.min_samples_split, '\n',
    'min_samples_leaf', best_rand_for_model.min_samples_leaf, '\n',
    'bootstrap', best_rand_for_model.bootstrap
    )

n_estimators 50 
max_depth 15 
 min_samples_splt 2 
 min_samples_leaf 1 
 bootstrap True


## Random Forest w/ GridSearch and SMOTE

In [142]:
# setting the initial hyperparameters
param_grid = {
    'n_estimators': [80, 90],
    'max_depth': [25, 30],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# setting the random forest model
rand_for = RandomForestClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=rand_for,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train_SMOTE, y_smote)
best_rand_for_model = grid_search.best_estimator_

# prediction
y_pred = best_rand_for_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Random Forest w/ GridSearch and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6640  218
1           1038   32

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.97      0.91      6858
           1       0.13      0.03      0.05      1070

    accuracy                           0.84      7928
   macro avg       0.50      0.50      0.48      7928
weighted avg       0.77      0.84      0.80      7928



In [143]:
print(
    'n_estimators', best_rand_for_model.n_estimators, '\n'
    'max_depth', best_rand_for_model.max_depth, '\n',
    'min_samples_splt', best_rand_for_model.min_samples_split, '\n',
    'min_samples_leaf', best_rand_for_model.min_samples_leaf, '\n',
    'bootstrap', best_rand_for_model.bootstrap
    )

n_estimators 90 
max_depth 30 
 min_samples_splt 2 
 min_samples_leaf 1 
 bootstrap False


## Storing Results

In [144]:
# all models
models = list(results.keys())

all_rows = []

for model in models:
    rep = results[model]  # classification_report dict

    row = {}
    row['model'] = model

    # classes 0 and 1
    for label in ['0', '1']:
        for metric in ['precision', 'recall', 'f1-score']:
            row[f'{label}_{metric}'] = rep[label][metric]

    # accuracy
    row['accuracy'] = rep['accuracy']

    # macro avg
    for metric in ['precision', 'recall', 'f1-score']:
        row[f'macro_{metric}'] = rep['macro avg'][metric]

    # weighted avg
    for metric in ['precision', 'recall', 'f1-score']:
        row[f'weighted_{metric}'] = rep['weighted avg'][metric]

    all_rows.append(row)

df_results = pd.DataFrame(all_rows)

df_results


,model,0_precision,0_recall,0_f1-score,1_precision,1_recall,1_f1-score,accuracy,macro_precision,macro_recall,macro_f1-score,weighted_precision,weighted_recall,weighted_f1-score
0,Logistic Regression,0.865035,1.000000,0.927634,0.000000,0.000000,0.000000,0.865035,0.432518,0.500000,0.463817,0.748286,0.865035,0.802436
1,Logistic Regression w/ Stepwise,0.865035,1.000000,0.927634,0.000000,0.000000,0.000000,0.865035,0.432518,0.500000,0.463817,0.748286,0.865035,0.802436
2,Logistic Regression w/ Stepwise and SMOTE,0.858877,0.412657,0.557471,0.130585,0.565421,0.212169,0.433274,0.494731,0.489039,0.384820,0.760583,0.433274,0.510867
3,KNN (24),0.865035,1.000000,0.927634,0.000000,0.000000,0.000000,0.865035,0.432518,0.500000,0.463817,0.748286,0.865035,0.802436
4,KNN (10) w/ Stepwise,0.865035,1.000000,0.927634,0.000000,0.000000,0.000000,0.865035,0.432518,0.500000,0.463817,0.748286,0.865035,0.802436
5,KNN (2) w/ Stepwise and SMOTE,0.864981,0.967775,0.913495,0.133333,0.031776,0.051321,0.841448,0.499157,0.499775,0.482408,0.766234,0.841448,0.797132
6,Decision Tree,0.867516,0.834500,0.850687,0.147258,0.183178,0.163265,0.746594,0.507387,0.508839,0.506976,0.770306,0.746594,0.757910
7,Decision Tree w/ GridSearch,0.864473,0.904054,0.883820,0.129630,0.091589,0.107338,0.794400,0.497051,0.497821,0.495579,0.765295,0.794400,0.779023
8,Decision Tree w/ GridSearch and SMOTE,0.868140,0.853456,0.860735,0.152614,0.169159,0.160461,0.761100,0.510377,0.511307,0.510598,0.771569,0.761100,0.766223
9,Random Forest w/ GridSearch,0.865035,1.000000,0.927634,0.000000,0.000000,0.000000,0.865035,0.432518,0.500000,0.463817,0.748286,0.865035,0.802436


# Final Prediction

## Data Treatment

In [ ]:
df_test = pd.read_csv('test.csv')
df_test = df_test.set_index('Unnamed: 0')
df_test = df_test.drop(columns=['data_batch_id']) #removing first column, that looks just an random id

# df_test = df_test.drop_duplicates() ????????????

# drop dates
df_test = df_test.drop(columns='application_date')

# removing unwanted object features
categ_cols = ['purpose_of_loan', 'employment_status', 'property_ownership_status']
df_test[categ_cols] = df_test[categ_cols].astype('category')
df_test = df_test.select_dtypes(exclude='object')

# capitalizing category features
for column in categ_cols:
    df_test[column] = df_test[column].str.capitalize()
    df_test[column] = df_test[column].str.strip()

# removing untrustable dummy columns
dummyDrop = [col for col in df_test.columns if 'loan_type' in col]
df_test = df_test.drop(columns=dummyDrop)

## Handling Missing Values and Unwanted Columns

### Outliers Removal

In [156]:
'''# removing outliers by the loan tenure months
q1 = df_test['loan_tenure_months'].quantile(0.25)
q3 = df_test['loan_tenure_months'].quantile(0.75)
iqr = q3-q1
upper_bound = q3 + iqr*1.5

# removing outliers
df_test = df_test[df_test['loan_tenure_months']>=upper_bound]'''

"# removing outliers by the loan tenure months\nq1 = df_test['loan_tenure_months'].quantile(0.25)\nq3 = df_test['loan_tenure_months'].quantile(0.75)\niqr = q3-q1\nupper_bound = q3 + iqr*1.5\n\n# removing outliers\ndf_test = df_test[df_test['loan_tenure_months']>=upper_bound]"

### Gender: Imputation

In [157]:
## gender imputation through KNN
def gender_code(row):
    if row['gender_Male'] == 1:
        return 1
    elif row['gender_Other'] == 1:
        return 0
    else:
        None #columns as neither male or other will be treated as unkown

df_test['gender_code'] = df_test.apply(gender_code, axis=1)

# defining known and unknown gender
df_known = df_test[df_test['gender_code'].notna()]
df_unknown = df_test[df_test['gender_code'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['gender_code']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_test.loc[df_test['gender_code'].isna(), 'gender_code'] = knn.predict(x_test_scaled)

# recreating the gender label, where "0" is man and "1" is woman
gender_drop = ['gender_Male', 'gender_Other']
df_test = df_test.drop(columns=gender_drop)
df_test = df_test.rename(columns={'gender_code': 'gender_male'})

# 0: woman ; 1: man
df_test['gender_male'].value_counts()

gender_male
0.0    5547
1.0    5453
Name: count, dtype: int64

### Number of Dependents: Imputation

In [158]:
# defining known and unknown number of dependents
df_known = df_test[df_test['number_of_dependents'].notna()]
df_unknown = df_test[df_test['number_of_dependents'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['number_of_dependents']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_test.loc[df_test['number_of_dependents'].isna(), 'number_of_dependents'] = knn.predict(x_test_scaled)

## Dropping

In [ ]:
# monthly income: removed since it has missing values and we already have the yearly income, with same effect
# cibil score: removed since it's perfectly normal, so likely it's not a real feature
# loan amount requested: removed since we have the same feature in dollars, to avoid redundance
df_test = df_test.drop(columns=['monthly_income', 'cibil_score', 'loan_amount_requested'])

## Dummying

In [160]:
# transforming categorical features into dummy ones, removing one of the categories to avoid colinearility
df_test = pd.get_dummies(df_test, columns=categ_cols, drop_first=True, dtype=int)

## Preparing Data for Modelling

In [161]:
# setting dummy columns
dummy_cols = [col for col in df_test if any(sub in col for sub in categ_cols) or col == 'gender_male']
not_dummy_cols = [col for col in df_test if col not in dummy_cols] 

# scaling feature columns but dummy ones
scaler = StandardScaler()
x_scaled = pd.DataFrame(
    scaler.fit_transform(df_test[not_dummy_cols]), #both train the model and scales
    columns=not_dummy_cols, 
    index=df_test.index)

# concatenating the dummy cols in the scaled numeric features
x_final = pd.concat([x_scaled, df_test[dummy_cols]], axis=1)

## Best Model: Logistic Regression w/ Stepwise

In [162]:
# DF test with stepwised columns
x_stepwised = x_final[x_stepwise_best_model]

# the smoted data should be used only for training, not for tests
df_test = df_test.copy()
df_test['fraud_flag'] = best_model.predict(x_stepwised)

## Exporting Prediction to Kaggle

In [163]:
df_test.index.name = "ID"
df_test['fraud_flag'].to_csv('submission_accuracy.csv', index=True)